# Performance Internals — 02: Adaptive Query Execution Deep Dive

## What you will learn
Adaptive Query Execution (AQE) was introduced in Spark 3.0 and is one of the most
impactful performance features in modern Spark. Unlike the static Catalyst optimizer
which works on estimated statistics, **AQE re-optimizes at runtime** using real data.

This notebook covers all three AQE optimizations with hands-on examples:
1. **Coalescing shuffle partitions** — avoiding the "too many small files" problem
2. **Converting SortMergeJoin to BroadcastHashJoin** — runtime size detection
3. **Skew join optimization** — splitting hot partitions automatically

## Why AQE matters
Without AQE, Spark plans queries based on **estimated statistics** which are often wrong:
- Table statistics may be stale or missing
- After a filter, actual row count may be much smaller than estimated
- Data distribution may be skewed in ways the optimizer cannot predict

AQE solves this by collecting real statistics at each shuffle boundary and replanning.


In [ ]:
import os, time, datetime
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

GLUTEN_ENABLED = os.environ.get('GLUTEN_ENABLED', 'false').lower() == 'true'
MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data'

spark = (
    SparkSession.builder
    .appName('performance-internals')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version} | Gluten: {GLUTEN_ENABLED}")

## Step 1 — Verify AQE Configuration

AQE is controlled by several settings. Let's check what we have configured.


In [ ]:
aqe_settings = [
    "spark.sql.adaptive.enabled",
    "spark.sql.adaptive.coalescePartitions.enabled",
    "spark.sql.adaptive.coalescePartitions.minPartitionSize",
    "spark.sql.adaptive.advisoryPartitionSizeInBytes",
    "spark.sql.adaptive.skewJoin.enabled",
    "spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes",
    "spark.sql.adaptive.skewJoin.skewedPartitionFactor",
    "spark.sql.autoBroadcastJoinThreshold",
]

print("Current AQE Configuration:")
print("-" * 60)
for s in aqe_settings:
    try:
        val = spark.conf.get(s)
        print(f"  {s.split('.')[-1]:<45} {val}")
    except Exception:
        print(f"  {s.split('.')[-1]:<45} (not set - using default)")

## Step 2 — AQE Feature 1: Coalescing Shuffle Partitions

After a shuffle, Spark creates `spark.sql.shuffle.partitions` partitions (default: 200).
For small datasets this means **200 tiny tasks** — massive overhead.

AQE coalescing merges adjacent small partitions into larger ones based on
`spark.sql.adaptive.advisoryPartitionSizeInBytes`.

**Problem without coalescing:**
- You have 1 MB of data after a shuffle
- With 200 partitions → each partition is 5 KB
- 200 tasks for 5 KB each = huge overhead

**With AQE coalescing:**
- Same 1 MB → merged into 1-2 partitions
- 1-2 tasks → much less overhead


In [ ]:
# Demonstrate partition coalescing
# Create a small dataset that after shuffle will have many tiny partitions

small_data = spark.range(10000).select(
    F.col("id"),
    (F.col("id") % 5).alias("group"),
    F.rand(42).alias("value")
)

# Without AQE coalescing
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "false")
spark.conf.set("spark.sql.shuffle.partitions", "200")

t0 = time.time()
result_no_coalesce = small_data.groupBy("group").agg(F.sum("value")).collect()
t_no_coalesce = time.time() - t0

# Check partition count
rdd_no_coalesce = small_data.groupBy("group").agg(F.sum("value")).rdd
print(f"Without AQE coalescing: {rdd_no_coalesce.getNumPartitions()} partitions, {t_no_coalesce:.3f}s")

# With AQE coalescing
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")

t0 = time.time()
result_coalesce = small_data.groupBy("group").agg(F.sum("value")).collect()
t_coalesce = time.time() - t0

rdd_coalesce = small_data.groupBy("group").agg(F.sum("value")).rdd
print(f"With AQE coalescing:    {rdd_coalesce.getNumPartitions()} partitions, {t_coalesce:.3f}s")
print()
print(f"AQE merged 200 partitions down to {rdd_coalesce.getNumPartitions()}")
print(f"Speedup: {t_no_coalesce/t_coalesce:.1f}x")

## Step 3 — AQE Feature 2: Dynamic Join Strategy

AQE can convert a `SortMergeJoin` to `BroadcastHashJoin` at runtime if one side
of the join turns out to be smaller than `autoBroadcastJoinThreshold` after filtering.

This is huge — the static optimizer may estimate a table as large (based on total size)
but after a filter the actual data is small enough to broadcast.


In [ ]:
# Create two tables where static stats say both are large
# but after filtering one becomes small

import random
random.seed(42)

# Large fact table: 50,000 orders
fact_data = [(i, random.randint(1, 1000), random.randint(1, 10),
              round(random.uniform(10, 500), 2))
             for i in range(50000)]
fact = spark.createDataFrame(fact_data, ["order_id","product_id","qty","price"])

# Dimension table: 1,000 products
dim_data = [(i, f"Product_{i}", random.choice(["A","B","C","D","E"]),
             round(random.uniform(5,500),2))
            for i in range(1, 1001)]
dim = spark.createDataFrame(dim_data, ["product_id","name","tier","cost"])

# Without filter: both tables are large → SortMergeJoin
join_no_filter = fact.join(dim, "product_id")
plan1 = join_no_filter._jdf.queryExecution().executedPlan().toString()
join_type1 = "BroadcastHashJoin" if "BroadcastHashJoin" in plan1 else "SortMergeJoin"

# With filter on dim: dim becomes tiny → AQE converts to BroadcastHashJoin
join_with_filter = fact.join(
    dim.filter(F.col("tier") == "A"),  # only ~200 rows after filter
    "product_id"
)

t0 = time.time(); join_no_filter.count(); t1 = time.time() - t0
t0 = time.time(); join_with_filter.count(); t2 = time.time() - t0

plan2 = join_with_filter._jdf.queryExecution().executedPlan().toString()
join_type2 = "BroadcastHashJoin" if "BroadcastHashJoin" in plan2 else "SortMergeJoin"

print(f"Without dim filter: {join_type1} ({t1:.3f}s)")
print(f"With dim filter   : {join_type2} ({t2:.3f}s)")
print()
print("AQE dynamically switched to BroadcastHashJoin after seeing the filtered dim size.")

## Step 4 — AQE Feature 3: Skew Join Optimization

Data skew is one of the most common Spark performance killers.
When one partition is 10x larger than others, that one task takes 10x longer
and all other executors sit idle waiting for it.

AQE detects skewed partitions and automatically **splits them** into smaller chunks,
which are then joined with replicated data from the other side.

A partition is considered skewed if:
- Its size > `skewedPartitionThresholdInBytes` (default: 256MB)
- AND its size > `skewedPartitionFactor` * median partition size (default: 5x)


In [ ]:
# Create a heavily skewed dataset
random.seed(42)

# 90% of data has key=0 (extreme skew)
N = 100000
skewed = spark.createDataFrame(
    [(i, 0 if random.random() < 0.9 else random.randint(1, 100),
      round(random.uniform(1, 100), 2))
     for i in range(N)],
    ["id", "key", "value"]
)

reference = spark.createDataFrame(
    [(i, f"ref_{i}") for i in range(101)],
    ["key", "label"]
)

# Check skew
key_dist = skewed.groupBy("key").count().orderBy(F.desc("count"))
print("Key distribution (top 5 — observe extreme skew on key=0):")
key_dist.show(5)

total = skewed.count()
skewed_count = skewed.filter(F.col("key") == 0).count()
print(f"Key 0 has {skewed_count:,} / {total:,} rows ({skewed_count/total*100:.0f}% of data)")

In [ ]:
# Join with skew — AQE will detect and split the skewed partition
skewed_join = skewed.join(reference, "key").groupBy("label").agg(F.sum("value"))

print("Plan with AQE skew join optimization:")
skewed_join.explain(mode="formatted")

t0 = time.time()
result = skewed_join.collect()
elapsed = time.time() - t0

print(f"\nCompleted in {elapsed:.3f}s")
print(f"Results: {len(result)} groups")
print("\nAQE split the skewed partition (key=0) into multiple sub-partitions.")
print("This distributes work evenly instead of one executor doing 90% of the work.")

## Step 5 — Measuring AQE Impact

Let's run a comprehensive benchmark comparing AQE on vs off.


In [ ]:
import time

def run_with_aqe(enabled, df_func, label):
    spark.conf.set("spark.sql.adaptive.enabled", str(enabled).lower())
    spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", str(enabled).lower())
    spark.conf.set("spark.sql.adaptive.skewJoin.enabled", str(enabled).lower())

    times = []
    for _ in range(3):
        t0 = time.time()
        df_func().collect()
        times.append(time.time() - t0)

    median = sorted(times)[1]
    print(f"  AQE {'ON ' if enabled else 'OFF'} | {label:<35} | median: {median:.3f}s")
    return median

print("Benchmark: AQE ON vs OFF")
print("-" * 65)

# Test 1: Small dataset aggregation (coalescing benefit)
q1 = lambda: spark.range(50000).select(
    (F.col("id") % 10).alias("g"), F.rand().alias("v")
).groupBy("g").agg(F.sum("v"))

# Test 2: Join with filter (dynamic broadcast benefit)
q2 = lambda: fact.join(dim.filter(F.col("tier") == "A"), "product_id") \
                  .groupBy("tier").agg(F.sum("price"))

t1_off = run_with_aqe(False, q1, "Small aggregation (200 partitions)")
t1_on  = run_with_aqe(True,  q1, "Small aggregation (200 partitions)")

t2_off = run_with_aqe(False, q2, "Join with selective filter")
t2_on  = run_with_aqe(True,  q2, "Join with selective filter")

# Restore AQE
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")

print()
print("AQE speedups:")
print(f"  Aggregation  : {t1_off/t1_on:.1f}x faster with AQE")
print(f"  Filtered join: {t2_off/t2_on:.1f}x faster with AQE")

## Summary: AQE Best Practices

### When AQE helps most
- **Variable data distributions** — some days have more data than others
- **After heavy filters** — actual data much smaller than table stats suggest
- **Complex multi-join queries** — AQE can reorder and replan at each stage
- **Skewed data** — any time one key dominates the dataset

### AQE limitations
- Cannot help with the **first stage** (before any shuffle) — no runtime stats yet
- Overhead of collecting statistics — negligible for large queries, visible for tiny ones
- Does not fix fundamentally bad query design (full table scans, cartesian joins)

### Configuration to tune
| Setting | Default | When to change |
|---|---|---|
| `advisoryPartitionSizeInBytes` | 64MB | Increase for large clusters |
| `skewedPartitionFactor` | 5 | Lower = more aggressive skew detection |
| `autoBroadcastJoinThreshold` | 10MB | Increase if you have > 10MB lookup tables |

**Next:** `03_memory_management.ipynb` — understanding executor memory, off-heap, and spill.
